# MCT Quickstart

Symbolic derivation → code generation → sampling, in 30 lines.

## Full Workflow

In [ ]:
import sympy as sp
from sympy import Matrix, sqrt, symbols
import mct
import numpy as np

# 1. Define Kraus operators
lam = symbols('lambda', real=True, positive=True)
K0 = Matrix([[1, 0], [0, sqrt(1 - lam)]])
K1 = Matrix([[0, sqrt(lam)], [0, 0]])

# 2. Derive symbolic transition matrix
P, meta = mct.derive_transition_matrix([K0, K1], mct.computational_basis, "T1")
mct.print_matrix_latex(P, "Symbolic P(λ)")

# 3. Verify at λ=0.5
P_num, valid = mct.substitute_and_verify(P, lam, 0.5, "Stochastic check")

# 4. Create MarkovChain
mc = mct.MarkovChain(P_num, state_labels=['|0⟩', '|1⟩'], name='T1')
print(f"MarkovChain valid: {mc.validate_stochasticity()['is_valid']}")

# 5. Generate sampling code
code = mct.markov_chain((P, meta), parameter_values={'lambda': 0.5})
print(f"Generated {len(code)} chars of sampling code")

$Symbolic P(λ) = \displaystyle \left[\begin{matrix}1 & \lambda\\0 & \sqrt{1 - \lambda} \overline{\sqrt{1 - \lambda}}\end{matrix}\right]$

Stochastic check
  λ = 0.5
  P_numeric =
[[1.  0.5]
 [0.  0.5]]
  Column sums: [1. 1.]
  ✓ Valid: True

MarkovChain valid: True
Generated 491 chars of sampling code


In [3]:
print(code)

def sample_markov_step(current_state: int) -> int:
    """Sample next state using cumulative probabilities."""
    import random
    r = random.random()

    if current_state == 0:  # 0
        if r < 1.000000000000000:
            return 0  # -> 0
        else:  # cumsum = 1.000000000000000
            return 1  # -> 1

    if current_state == 1:  # 1
        if r < 0.500000000000000:
            return 0  # -> 0
        else:  # cumsum = 1.000000000000000
            return 1  # -> 1

